# Quantitative Analysis: Serbian Social Card Registry Audit

**Eticas Foundation — Community-Led Audit**  
**Data source:** Micro-dataset compiled by A11 – Initiative for Economic and Social Rights  
**N = 15 cases**

This notebook reproduces the descriptive statistics reported in the Audit Report (Section 2: Empirical Testing and Quantitative Impact Assessment). It covers:

1. Sample demographics (age, gender, household size)
2. Exclusion category frequencies
3. Employment status distribution
4. Geographic distribution
5. Due process violations (exclusions without written decisions)

---

> ⚠️ **Limitation notice:** This dataset is not a representative sample. Findings support indicative evidence only and cannot be generalised to the broader population of Social Card Registry beneficiaries. See `data/README.md` for full provenance and limitations.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Eticas brand-adjacent palette (neutral, accessible)
COLORS = {
    'primary': '#1a1a2e',
    'accent': '#e94560',
    'mid': '#16213e',
    'light': '#0f3460',
    'muted': '#a8a8b3',
    'vehicle': '#e07b39',
    'income': '#e94560',
    'seasonal': '#4361ee',
    'due_process': '#7b2d8b',
}

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'sans-serif',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

df = pd.read_csv('../data/micro-dataset.csv')
print(f"Dataset loaded: {len(df)} cases")
df.head()

## 1. Sample Demographics

In [ ]:
n = len(df)
n_female = (df['gender'] == 'F').sum()
n_male = (df['gender'] == 'M').sum()
pct_female = n_female / n * 100

mean_age = df['age'].mean()
median_age = df['age'].median()
median_household = df['household_size'].median()
mean_household = df['household_size'].mean()

print("=== SAMPLE DEMOGRAPHICS ===")
print(f"Total cases (N):         {n}")
print(f"Female:                  {n_female} ({pct_female:.0f}%)")
print(f"Male:                    {n_male} ({100-pct_female:.0f}%)")
print(f"Mean age:                {mean_age:.1f} years")
print(f"Median age:              {median_age:.0f} years")
print(f"Age range:               {df['age'].min()}–{df['age'].max()} years")
print(f"Median household size:   {median_household:.0f} members")
print(f"Mean household size:     {mean_household:.1f} members")
print(f"Household size range:    {df['household_size'].min()}–{df['household_size'].max()} members")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Sample Demographics (N=15)', fontsize=14, fontweight='bold', y=1.02)

# Gender
gender_counts = df['gender'].value_counts()
axes[0].pie(
    gender_counts.values,
    labels=['Female (53%)', 'Male (47%)'],
    colors=[COLORS['accent'], COLORS['light']],
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[0].set_title('Gender Distribution')

# Age
axes[1].hist(df['age'], bins=8, color=COLORS['light'], edgecolor='white', linewidth=0.8)
axes[1].axvline(mean_age, color=COLORS['accent'], linestyle='--', linewidth=1.5, label=f'Mean: {mean_age:.1f}')
axes[1].set_xlabel('Age (years)')
axes[1].set_ylabel('Count')
axes[1].set_title('Age Distribution')
axes[1].legend(fontsize=9)

# Household size
axes[2].hist(df['household_size'], bins=range(1, 13), color=COLORS['primary'], edgecolor='white', linewidth=0.8, align='left')
axes[2].axvline(median_household, color=COLORS['accent'], linestyle='--', linewidth=1.5, label=f'Median: {median_household:.0f}')
axes[2].set_xlabel('Household size (members)')
axes[2].set_ylabel('Count')
axes[2].set_title('Household Size Distribution')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.savefig('fig1_demographics.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig1_demographics.png")

## 2. Exclusion Category Frequencies

Cases were coded into three analytical categories (non-mutually exclusive) corresponding to the audit's harm hypotheses.

In [ ]:
cat_vehicle = df['exclusion_category_vehicle'].sum()
cat_income = df['exclusion_category_income_misclassification'].sum()
cat_seasonal = df['exclusion_category_seasonal_work'].sum()
cat_no_decision = df['exclusion_without_written_decision'].sum()

print("=== EXCLUSION CATEGORIES ===")
print(f"Vehicle-related flags:              {cat_vehicle}/{n} ({cat_vehicle/n*100:.0f}%)")
print(f"Income misclassification/donations: {cat_income}/{n} ({cat_income/n*100:.0f}%)  ← Accuracy Hypothesis")
print(f"Seasonal work income:               {cat_seasonal}/{n} ({cat_seasonal/n*100:.0f}%)  ← Compliance Hypothesis")
print(f"Action without written decision:    {cat_no_decision}/{n} ({cat_no_decision/n*100:.0f}%)  ← Transparency Hypothesis")
print()
print("Note: categories are non-mutually exclusive (a case may appear in more than one)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart of category frequencies
categories = [
    'Income\nMisclassification\n(Accuracy)',
    'Seasonal Work\nIncome\n(Compliance)',
    'Vehicle\nFlags',
    'No Written\nDecision\n(Transparency)',
]
counts = [cat_income, cat_seasonal, cat_vehicle, cat_no_decision]
pcts = [c / n * 100 for c in counts]
bar_colors = [COLORS['income'], COLORS['seasonal'], COLORS['vehicle'], COLORS['due_process']]

bars = axes[0].barh(categories, pcts, color=bar_colors, edgecolor='white', height=0.5)
for bar, count, pct in zip(bars, counts, pcts):
    axes[0].text(
        bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
        f'{count}/{n} ({pct:.0f}%)', va='center', fontsize=10
    )
axes[0].set_xlim(0, 80)
axes[0].set_xlabel('% of cases (N=15)')
axes[0].set_title('Exclusion Categories by Frequency\n(non-mutually exclusive)', fontweight='bold')

# Overlap: cases with multiple categories
df['n_categories'] = (
    df['exclusion_category_vehicle'].astype(int) +
    df['exclusion_category_income_misclassification'].astype(int) +
    df['exclusion_category_seasonal_work'].astype(int)
)
overlap_counts = df['n_categories'].value_counts().sort_index()
axes[1].bar(
    [f'{i} category' if i == 1 else f'{i} categories' for i in overlap_counts.index],
    overlap_counts.values,
    color=[COLORS['light'], COLORS['accent'], COLORS['muted']],
    edgecolor='white'
)
for i, (x, y) in enumerate(zip(overlap_counts.index, overlap_counts.values)):
    label = f'{x} category' if x == 1 else f'{x} categories'
    axes[1].text(i, y + 0.1, str(y), ha='center', fontsize=11)
axes[1].set_ylabel('Number of cases')
axes[1].set_title('Cases by Number of\nExclusion Categories', fontweight='bold')
axes[1].set_ylim(0, max(overlap_counts.values) + 1.5)

plt.tight_layout()
plt.savefig('fig2_exclusion_categories.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig2_exclusion_categories.png")

## 3. Employment Status Distribution

In [ ]:
emp_counts = df['employment_status'].value_counts()
print("=== EMPLOYMENT STATUS ===")
for status, count in emp_counts.items():
    print(f"  {status}: {count} ({count/n*100:.0f}%)")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

emp_palette = [COLORS['light'], COLORS['accent'], COLORS['vehicle'], COLORS['muted']]
bars = ax.bar(
    emp_counts.index, emp_counts.values,
    color=emp_palette[:len(emp_counts)],
    edgecolor='white'
)
for bar in bars:
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
        str(int(bar.get_height())), ha='center', fontsize=11
    )
ax.set_ylabel('Number of cases')
ax.set_title('Employment Status at Time of Exclusion (N=15)', fontweight='bold')
ax.set_ylim(0, emp_counts.max() + 1.5)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig('fig3_employment_status.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig3_employment_status.png")

## 4. Geographic Distribution & Exclusion Patterns by Location

In [ ]:
loc_counts = df['location'].value_counts()
print("=== GEOGRAPHIC DISTRIBUTION ===")
for loc, count in loc_counts.items():
    print(f"  {loc}: {count} cases")

# Dominant exclusion type by location
print("\n=== DOMINANT EXCLUSION TYPE BY LOCATION ===")
for loc in df['location'].unique():
    subset = df[df['location'] == loc]
    seasonal = subset['exclusion_category_seasonal_work'].sum()
    income = subset['exclusion_category_income_misclassification'].sum()
    vehicle = subset['exclusion_category_vehicle'].sum()
    print(f"  {loc} (n={len(subset)}): seasonal={seasonal}, income_misclass={income}, vehicle={vehicle}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Case count by city
axes[0].barh(loc_counts.index, loc_counts.values, color=COLORS['light'], edgecolor='white')
for i, v in enumerate(loc_counts.values):
    axes[0].text(v + 0.1, i, str(v), va='center', fontsize=11)
axes[0].set_xlabel('Number of cases')
axes[0].set_title('Cases by Location', fontweight='bold')
axes[0].set_xlim(0, loc_counts.max() + 1.5)

# Stacked exclusion categories by location
locations = df['location'].unique()
seasonal_by_loc = [df[df['location']==l]['exclusion_category_seasonal_work'].sum() for l in locations]
income_by_loc = [df[df['location']==l]['exclusion_category_income_misclassification'].sum() for l in locations]
vehicle_by_loc = [df[df['location']==l]['exclusion_category_vehicle'].sum() for l in locations]

x = range(len(locations))
axes[1].bar(x, income_by_loc, label='Income misclassification', color=COLORS['income'], edgecolor='white')
axes[1].bar(x, seasonal_by_loc, bottom=income_by_loc, label='Seasonal work', color=COLORS['seasonal'], edgecolor='white')
axes[1].bar(x, vehicle_by_loc,
           bottom=[i+s for i,s in zip(income_by_loc, seasonal_by_loc)],
           label='Vehicle flags', color=COLORS['vehicle'], edgecolor='white')
axes[1].set_xticks(x)
axes[1].set_xticklabels(locations, rotation=15, ha='right')
axes[1].set_ylabel('Number of cases (non-exclusive)')
axes[1].set_title('Exclusion Categories by Location\n(non-mutually exclusive)', fontweight='bold')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('fig4_geographic.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig4_geographic.png")

## 5. Due Process Violations

Cases where benefit was suspended or reduced without a written administrative decision, in violation of beneficiaries' right to appeal.

In [ ]:
no_decision = df[df['exclusion_without_written_decision'] == True][['initials', 'location', 'age', 'gender', 'exclusion_reason_raw']]
print(f"Cases without written decision: {len(no_decision)}/{n} ({len(no_decision)/n*100:.0f}%)")
print()
print(no_decision.to_string(index=False))

## 6. Summary Table — Harm Hypothesis Validation

Reproduces Table 1 from the Audit Report, with quantitative backing.

In [ ]:
summary = pd.DataFrame({
    'Harm Hypothesis': [
        'Accuracy: Income/asset misclassification',
        'Compliance: Seasonal work income counted',
        'Transparency: Action without written decision',
        'Discrimination proxy: Vehicle flags (disproportionate in informal/Roma contexts)',
    ],
    'Cases (n)': [cat_income, cat_seasonal, cat_no_decision, cat_vehicle],
    'Cases (%)': [
        f'{cat_income/n*100:.0f}%',
        f'{cat_seasonal/n*100:.0f}%',
        f'{cat_no_decision/n*100:.0f}%',
        f'{cat_vehicle/n*100:.0f}%',
    ],
    'Audit Finding': [
        'Supports Accuracy Hypothesis',
        'Supports Compliance Hypothesis',
        'Supports Transparency Hypothesis',
        'Indicative; broader data needed',
    ]
})

print("=== HARM HYPOTHESIS VALIDATION SUMMARY ===")
print(summary.to_string(index=False))
summary

---

## Methodology Note

The micro-dataset (N=15) was constructed by coding the shared A11 table into three analytical categories:

- **Vehicle-related** (`exclusion_category_vehicle`): any case where vehicle ownership or outdated vehicle registration contributed to the exclusion trigger.
- **Income misclassification/donations** (`exclusion_category_income_misclassification`): any case where income was inflated, incorrectly calculated, or where a non-income event (e.g. funeral donation) was treated as income.
- **Seasonal work** (`exclusion_category_seasonal_work`): any case where seasonal agricultural wages — legally exempt from means-testing under the Law on Seasonal Workers — were treated as disqualifying income.

Categories are **non-mutually exclusive**. A case may be coded TRUE in more than one category.

The variable `exclusion_without_written_decision` captures cases where benefit suspension or reduction occurred without a formal written administrative decision, as explicitly noted in A11's original case records.

All analysis uses **descriptive statistics only** (counts, percentages, means, medians). The sample size precludes hypothesis testing or inferential statistics.